# DeepSeek Harness 教学 Notebook

这套 Notebook 对应 `deepseek_harness/`，重点观察：

- `DEEPSEEK.md` / `AGENTS.md` 如何进入上下文；
- `reasoning_content` 如何与最终答案分离；
- 模型失败如何有限重试；
- 工具错误如何作为 `ToolMessage` 回填；
- 每一步如何写 checkpoint。


In [ ]:
from pathlib import Path
import json, shutil, tempfile

from deepseek_harness.config import DeepSeekHarnessConfig
from deepseek_harness.context import DeepSeekContextManager
from deepseek_harness.harness import DeepSeekHarness


## 1. 上下文分层

先只看上下文，不调用模型。

In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="deepseek-nb-"))
shutil.copytree("examples/deepseek_harness_demo", workspace, dirs_exist_ok=True)
ctx = DeepSeekContextManager(workspace, model_name="deepseek-v4.1-flash")
snapshot = ctx.snapshot("修复 calculator.py 的 bug")
for key, value in snapshot.items():
    print(f"===== {key} =====")
    print(value[:800])


## 2. 运行完整 Harness

需要 `.env` 中配置 `BAILIAN_API_KEY`、`BAILIAN_BASE_URL` 和 DeepSeek 模型名。

In [ ]:
config = DeepSeekHarnessConfig(
    workspace=workspace,
    max_steps=8,
    verify_command="python -m pytest -q",
    console=False,
)
result = DeepSeekHarness(config).run("修复 calculator.py 中 add 的 bug，运行测试并总结")
print(result.final_answer)
print("success=", result.success, "checkpoints=", result.checkpoints, "retries=", result.retries)


## 3. 逐个事件内视

事件流正是前端 `Mini Harness Studio` 读取的数据源。

In [ ]:
events = [json.loads(line) for line in (result.run_dir / "events.jsonl").read_text(encoding="utf-8").splitlines()]
for event in events:
    print(f"{event['type']:24s} {event['stage']:10s} {event['message'][:100]}")


## 4. 查看 checkpoint

checkpoint 在模型调用后和工具执行后都会更新，用来定位失败断点。

In [ ]:
checkpoint = json.loads((result.run_dir / "checkpoint.json").read_text(encoding="utf-8"))
print("step:", checkpoint["step"], "phase:", checkpoint["phase"])
print("messages:", len(checkpoint["messages"]))
print("changed_files:", checkpoint["changed_files"])
